In [2]:
# ── 1 · Импорты, конфиг, логирование ─────────────────────────────
import os, logging, importlib.util
import numpy as np
import pandas as pd
import databento as db

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('imkt')

PATH_NQ_ES = './data/GLBX-20260603-XBMUUNEJJQ 16yr nq es data/glbx-mdp3-20100606-20260602.ohlcv-1m.dbn'
CACHE_DIR  = './cache'
LAB_PATH   = './intermarket_lab.py'
os.makedirs(CACHE_DIR, exist_ok=True)
log.info('OK')

19:31:35  INFO      OK


In [3]:
# ── 2 · VIX (FRED, с кешем) ───────────────────────────────────────
VIX_CACHE = os.path.join(CACHE_DIR, 'vix_daily.parquet')
if os.path.exists(VIX_CACHE):
    vix = pd.read_parquet(VIX_CACHE)
    log.info('VIX из кеша: %d дней', len(vix))
else:
    from fredapi import Fred
    fred = Fred(api_key='YOUR_FRED_API_KEY')
    vix = (fred.get_series('VIXCLS', observation_start='2010-01-01',
                           observation_end='2026-06-01')
           .to_frame('vix_close'))
    vix.index = pd.to_datetime(vix.index); vix = vix.dropna()
    vix.to_parquet(VIX_CACHE)
    log.info('VIX через FRED: %d дней', len(vix))

19:31:35  INFO      VIX из кеша: 4157 дней


In [4]:
# ── 3 · Макро: ZN, ZT, CL, DX (+ DX FRED до 2018) ────────────────
MACRO_PATHS = {
    'zn': './data/GLBX-20260604-YY86KWTSMD 10yr/glbx-mdp3-20100606-20260605.ohlcv-1d.dbn',
    'zt': './data/GLBX-20260606-BTSF5RNAY9 zt 2yr/glbx-mdp3-20100606-20260605.ohlcv-1d.dbn',
    'cl': './data/GLBX-20260606-EWECGEANF7 cl/glbx-mdp3-20100606-20260605.ohlcv-1d.dbn',
    'dx': './data/IFUS-20260606-3B38JP5FG8 2 dx 2018/ifus-impact-20181223-20260605.ohlcv-1d.dbn',
}
def load_macro(name, path, symbol_filter=None, cache_dir=CACHE_DIR):
    cache = os.path.join(cache_dir, f'{name}_daily.parquet')
    if os.path.exists(cache):
        log.info('%s из кеша', name.upper()); return pd.read_parquet(cache)
    store = db.DBNStore.from_file(path); df = store.to_df()
    df.index = pd.to_datetime(df.index).tz_localize(None)
    if symbol_filter:
        df = df[df['symbol'].str.startswith(symbol_filter) & ~df['symbol'].str.contains('-')].copy()
    df['date'] = df.index.date
    if 'symbol' in df.columns and df['symbol'].nunique() > 1:
        dv = df.groupby(['date','symbol'])['volume'].sum()
        front = dv.groupby('date').idxmax().apply(lambda x: x[1])
        df['front'] = df['date'].map(front); df = df[df['symbol'] == df['front']]
    daily = df.groupby('date').agg(open=('open','first'), high=('high','max'),
        low=('low','min'), close=('close','last'), volume=('volume','sum'))
    daily.index = pd.to_datetime(daily.index); daily.to_parquet(cache)
    log.info('%s построен: %d дней', name.upper(), len(daily)); return daily

zn = load_macro('zn', MACRO_PATHS['zn'], 'ZN')
zt = load_macro('zt', MACRO_PATHS['zt'], 'ZT')
cl = load_macro('cl', MACRO_PATHS['cl'], 'CL')
dx = load_macro('dx', MACRO_PATHS['dx'])

DX_FRED_CACHE = os.path.join(CACHE_DIR, 'dx_fred.parquet')
if not os.path.exists(DX_FRED_CACHE):
    from fredapi import Fred
    fred = Fred(api_key='YOUR_FRED_API_KEY')
    dx_fred = fred.get_series('DTWEXBGS', observation_start='2010-01-01',
                              observation_end='2018-12-22').to_frame('close')
    dx_fred.index = pd.to_datetime(dx_fred.index); dx_fred.to_parquet(DX_FRED_CACHE)
else:
    dx_fred = pd.read_parquet(DX_FRED_CACHE)
dx_full = pd.concat([dx_fred[['close']], dx[['close']]]).sort_index()
dx_full = dx_full[~dx_full.index.duplicated(keep='last')]

macro = pd.concat([zn['close'].rename('zn_close'), zt['close'].rename('zt_close'),
                   cl['close'].rename('cl_close'), dx_full['close'].rename('dx_close')], axis=1)
macro['yield_slope'] = macro['zn_close'] - macro['zt_close']
log.info('Macro готов: %d дней', len(macro))

19:31:35  INFO      ZN из кеша
19:31:35  INFO      ZT из кеша
19:31:35  INFO      CL из кеша
19:31:35  INFO      DX из кеша
19:31:35  INFO      Macro готов: 5097 дней


In [5]:
# ── 4 · ES/NQ continuous (1m, с кешем) ───────────────────────────
ES_CONT = os.path.join(CACHE_DIR, 'es_continuous.parquet')
NQ_CONT = os.path.join(CACHE_DIR, 'nq_continuous.parquet')
if os.path.exists(ES_CONT) and os.path.exists(NQ_CONT):
    es_continuous = pd.read_parquet(ES_CONT); nq_continuous = pd.read_parquet(NQ_CONT)
    log.info('Continuous из кеша  ES %s  NQ %s', f'{len(es_continuous):,}', f'{len(nq_continuous):,}')
else:
    store = db.DBNStore.from_file(PATH_NQ_ES); df = store.to_df()
    df.index = df.index.tz_localize(None)
    fs = [s for s in df['symbol'].unique() if '-' not in s]
    dfc = df[df['symbol'].isin(fs)].copy()
    es = dfc[dfc['symbol'].str.startswith('ES')].copy()
    nq = dfc[dfc['symbol'].str.startswith('NQ')].copy()
    def front(d):
        d = d.copy(); d['date'] = d.index.date
        return d.groupby(['date','symbol'])['volume'].sum().groupby('date').idxmax().apply(lambda x: x[1])
    def cont(d, fc):
        d = d.copy(); d['date'] = d.index.date; d['front'] = d['date'].map(fc)
        return d[d['symbol'] == d['front']].drop(columns=['date','front']).sort_index()
    es_continuous = cont(es, front(es)); nq_continuous = cont(nq, front(nq))
    es_continuous.to_parquet(ES_CONT); nq_continuous.to_parquet(NQ_CONT)
log.info('ES %s..%s', es_continuous.index[0].date(), es_continuous.index[-1].date())

19:31:35  INFO      Continuous из кеша  ES 5,586,072  NQ 5,414,367
19:31:35  INFO      ES 2010-06-06..2026-06-02


In [6]:
# ── 5 · RTH-фильтр (14:30–21:00 UTC) ─────────────────────────────
RTH_START, RTH_END = 14*60+30, 21*60+0
def get_rth(d):
    t = d.index.hour*60 + d.index.minute
    return d[(t >= RTH_START) & (t < RTH_END)].copy()
es_rth = get_rth(es_continuous); nq_rth = get_rth(nq_continuous)
log.info('RTH  ES %s  NQ %s', f'{len(es_rth):,}', f'{len(nq_rth):,}')

19:31:36  INFO      RTH  ES 1,544,264  NQ 1,541,888


In [7]:
# ── 6 · L0 · intermarket/macro фичи (модуль intermarket_lab) ─────
spec = importlib.util.spec_from_file_location('intermarket_lab', LAB_PATH)
il = importlib.util.module_from_spec(spec); spec.loader.exec_module(il)

FEATURES      = il.INTERMARKET_FEATURES
N_REGIMES     = 7
COV_TYPE      = 'diag'
DISCOVERY_END = '2019-01-01'
WARMUP_MONTHS = 24
WF_NINIT      = 4

feat = il.build_features(es_rth, nq_rth, vix, macro, zn, cl)
_c = feat[FEATURES].dropna()
log.info('L0: %d intermarket-фич, полных строк %d из %d (%.0f%%)',
         len(FEATURES), len(_c), len(feat), 100*len(_c)/len(feat))

intermarket_lab OK


19:31:37  INFO      L0: 18 intermarket-фич, полных строк 3280 из 4106 (80%)


In [8]:
# ── 7 · L1 · эталонные центроиды режимов на discovery ────────────
REF_CENTROIDS = il.build_reference(feat[feat.index < DISCOVERY_END], FEATURES, N_REGIMES)
log.info('L1: эталон %d режимов на discovery (<%s)', N_REGIMES, DISCOVERY_END)

19:31:37  INFO      L1: эталон 7 режимов на discovery (<2019-01-01)


In [9]:
# ── 8 · L2 · expanding WF → OOS-леджер (next-day open→close) ──────
_months = pd.period_range(feat.index.min(), feat.index.max(), freq='M')
_first  = feat.index.min() + pd.DateOffset(months=WARMUP_MONTHS)
_blk = []
for per in _months:
    m0 = per.to_timestamp(); m1 = (per+1).to_timestamp()
    if m0 < _first: continue
    train = feat[feat.index < m0]
    test  = feat[(feat.index >= m0) & (feat.index < m1)]
    if len(test) == 0 or len(train) < 252: continue
    sc, gmm = il.fit_regime_model(train, FEATURES, N_REGIMES, n_init=WF_NINIT, cov_type=COV_TYPE)
    reg, regc = il.assign_regimes(test, sc, gmm, il.canonical_map(sc, gmm, REF_CENTROIDS), FEATURES)
    _blk.append(pd.DataFrame({'regime': reg, 'regime_conf': regc}, index=test.index))

wf = pd.concat(_blk).sort_index().join(feat[['open_px','close_px','fwd_ret']])
wf['regime_prev']      = wf['regime'].shift(1)
wf['regime_conf_prev'] = wf['regime_conf'].shift(1)
wf['win']     = wf['fwd_ret'] > 0
wf['abs_ret'] = wf['fwd_ret'].abs()
wf = wf.dropna(subset=['regime_prev','fwd_ret'])
log.info('L2 OOS-леджер: %d дней %s..%s  baseline WR=%.1f%% avg=%.3f%% |move|=%.3f%%',
         len(wf), wf.index.min().date(), wf.index.max().date(),
         wf['win'].mean()*100, wf['fwd_ret'].mean()*100, wf['abs_ret'].mean()*100)
log.info('Режимы (prev):\n%s', wf['regime_prev'].astype(int).value_counts().sort_index().to_string())

19:31:45  INFO      L2 OOS-леджер: 3070 дней 2012-07-03..2026-06-02  baseline WR=55.5% avg=0.020% |move|=0.482%
19:31:45  INFO      Режимы (prev):
regime_prev
0    671
1    576
2    695
3    280
4    147
5    395
6    306


In [10]:
# ── 9 · L3 · эдж по режимам: направление (t / MW / WR, FDR) ──────
from scipy import stats
MIN_N, Q = 40, 0.10
POOL_MEAN, POOL_WR = wf['fwd_ret'].mean(), wf['win'].mean()
POOL_W, POOL_N = int(wf['win'].sum()), len(wf)

def _edge(col, y):
    rows = []
    for k, g in wf.groupby(col):
        n = len(g)
        if n < MIN_N: continue
        kk = int(g['win'].sum()); rest = wf[y].drop(g.index)
        _, pt  = stats.ttest_ind(g[y], rest, equal_var=False)
        _, pmw = stats.mannwhitneyu(g[y], rest, alternative='two-sided')
        rows.append({'regime': int(k), 'n': n, 'wr': round(g['win'].mean()*100,1),
            'wr_lb': round(il.wilson_lb(kk,n)*100,1), 'mean': round(g[y].mean()*100,3),
            'p_t': pt, 'p_mw': pmw, 'p_wr': il.two_prop_p(kk,n,POOL_W-kk,POOL_N-n)})
    d = pd.DataFrame(rows)
    for c in ['p_t','p_mw','p_wr']: d['q'+c[1:]] = il.bh_fdr(d[c].values)
    return d.sort_values('p_mw')

edge_dir = _edge('regime_prev', 'fwd_ret')
log.info('L3 НАПРАВЛЕНИЕ (baseline WR=%.1f%%), значимых q<%.2f: t=%d MW=%d WR=%d\n%s',
         POOL_WR*100, Q, int((edge_dir['q_t']<Q).sum()), int((edge_dir['q_mw']<Q).sum()),
         int((edge_dir['q_wr']<Q).sum()), edge_dir.to_string(index=False))

19:31:45  INFO      L3 НАПРАВЛЕНИЕ (baseline WR=55.5%), значимых q<0.10: t=0 MW=0 WR=0
 regime   n   wr  wr_lb   mean      p_t     p_mw     p_wr      q_t     q_mw     q_wr
      3 280 59.3   53.4  0.064 0.511379 0.042148 0.185506 0.651862 0.295036 0.519901
      4 147 57.8   49.7  0.059 0.558739 0.152400 0.567642 0.651862 0.468767 0.721085
      2 695 52.5   48.8 -0.005 0.268634 0.200900 0.068568 0.651862 0.468767 0.479976
      0 671 56.2   52.4 -0.005 0.227877 0.276515 0.702661 0.651862 0.483901 0.721085
      1 576 57.8   53.7  0.042 0.283809 0.431337 0.222815 0.651862 0.603871 0.519901
      5 395 53.7   48.7  0.011 0.818610 0.625494 0.423849 0.818610 0.729743 0.721085
      6 306 54.6   49.0  0.041 0.508179 0.868972 0.721085 0.651862 0.868972 0.721085


In [11]:
# ── 10 · L3-vol · эдж по режимам: магнитуда |fwd_ret| ────────────
def _edge_vol(col):
    rows = []
    for k, g in wf.groupby(col):
        n = len(g)
        if n < MIN_N: continue
        rest = wf['abs_ret'].drop(g.index)
        _, pt  = stats.ttest_ind(g['abs_ret'], rest, equal_var=False)
        _, pmw = stats.mannwhitneyu(g['abs_ret'], rest, alternative='two-sided')
        rows.append({'regime': int(k), 'n': n, 'abs_ret': round(g['abs_ret'].mean()*100,3),
            'lift': round((g['abs_ret'].mean()-wf['abs_ret'].mean())*100,3), 'p_t': pt, 'p_mw': pmw})
    d = pd.DataFrame(rows); d['q_mw'] = il.bh_fdr(d['p_mw'].values)
    return d.sort_values('p_mw')

edge_vol = _edge_vol('regime_prev')
log.info('L3-vol МАГНИТУДА (baseline |ret|=%.3f%%), значимых q_mw<%.2f: %d\n%s',
         wf['abs_ret'].mean()*100, Q, int((edge_vol['q_mw']<Q).sum()), edge_vol.to_string(index=False))

19:31:45  INFO      L3-vol МАГНИТУДА (baseline |ret|=0.482%), значимых q_mw<0.10: 7
 regime   n  abs_ret   lift          p_t         p_mw         q_mw
      3 280    0.802  0.320 1.021789e-09 5.594115e-16 3.915881e-15
      0 671    0.376 -0.106 2.074402e-12 1.069355e-09 3.742742e-09
      1 576    0.358 -0.124 1.214035e-15 2.204706e-09 5.144314e-09
      5 395    0.629  0.148 2.424382e-07 8.724248e-09 1.526743e-08
      6 306    0.402 -0.079 5.385478e-04 1.228561e-02 1.455860e-02
      4 147    0.598  0.117 1.402607e-02 1.247880e-02 1.455860e-02
      2 695    0.484  0.002 9.009439e-01 5.160783e-02 5.160783e-02


In [12]:
# ── 11 · Vol-model · непрерывный intermarket-предиктор (главный) ──
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr

INTER = feat[FEATURES].shift(1)   # фичи дня t-1 → предсказываем день t

def _nested_oos(blocks, y):
    idx = y.dropna().index
    for _, X in blocks: idx = idx.intersection(X.dropna().index)
    months = pd.period_range(idx.min(), idx.max(), freq='M')
    out = {}
    for name, X in blocks:
        Xv, yv = X.loc[idx], y.loc[idx]; pred = pd.Series(np.nan, index=idx)
        for per in months:
            m0 = per.to_timestamp(); m1 = (per+1).to_timestamp()
            tr = Xv.index[Xv.index < m0]; te = Xv.index[(Xv.index >= m0) & (Xv.index < m1)]
            if len(tr) < 252 or len(te) == 0: continue
            sc = StandardScaler().fit(Xv.loc[tr])
            rg = Ridge(alpha=1.0).fit(sc.transform(Xv.loc[tr]), yv.loc[tr])
            pred.loc[te] = rg.predict(sc.transform(Xv.loc[te]))
        m = pred.dropna().index; yt, pp = yv.loc[m], pred.loc[m]
        sse = ((yt-pp)**2).sum(); sst = ((yt-yt.mean())**2).sum()
        out[name] = {'R2': 1-sse/sst, 'Spearman': spearmanr(pp,yt).correlation, 'n': len(m)}
    return out

# VOL (главный продукт)
y_vol  = wf['abs_ret']
nv_vol = y_vol.ewm(halflife=10).mean().shift(1).rename('ewma').to_frame()
rv = _nested_oos([('M0_naive', nv_vol),
                  ('M1_+intermarket', pd.concat([nv_vol, INTER], axis=1))], y_vol)
log.info('VOL OOS-предсказание:\n%s', pd.DataFrame(rv).T.round(4).to_string())
log.info('  ΔR² intermarket: %+.4f', rv['M1_+intermarket']['R2'] - rv['M0_naive']['R2'])

# DIRECTION (проверка — ожидаемо слабо)
y_dir  = wf['fwd_ret']
nv_dir = y_dir.shift(1).rename('mom').to_frame()
rd = _nested_oos([('M0_naive', nv_dir),
                  ('M1_+intermarket', pd.concat([nv_dir, INTER], axis=1))], y_dir)
log.info('DIRECTION OOS-предсказание:\n%s', pd.DataFrame(rd).T.round(4).to_string())

19:31:46  INFO      VOL OOS-предсказание:
                     R2  Spearman       n
M0_naive         0.1165    0.3228  2813.0
M1_+intermarket  0.1711    0.3679  2813.0
19:31:46  INFO        ΔR² intermarket: +0.0546
19:31:46  INFO      DIRECTION OOS-предсказание:
                     R2  Spearman       n
M0_naive         0.0013    0.0152  2813.0
M1_+intermarket -0.0093    0.0370  2813.0
